In [0]:
from pyspark.sql import functions as F

Load the bronze table

In [0]:
bronze_df = spark.table("energy.bronze.consumption")

In [0]:
display(bronze_df)

Create the silver table and clean the data


In [0]:
pre_silver_df = (bronze_df.select(
    F.col("datasetId").cast("int").alias("dataset_id"),
    F.col("startTime").cast("timestamp").alias("start_time"),
    F.col("endTime").cast("timestamp").alias("end_time"),
    F.col("value")
        .cast("double")
        .alias("consumption_mw")
))

In [0]:
display(pre_silver_df)

Validate the data


In [0]:
validated_df = (
    pre_silver_df
    .withColumn(
        "validation_error",
        F.when(
            F.col("dataset_id").isNull(),
            "missing_dataset_id"
        )
        .when(
            F.col("start_time").isNull(),
            "missing_start_time"
        )
        .when(
            F.col("end_time").isNull(),
            "missing_end_time"
        )
        .when(
            F.col("consumption_mw").isNull(),
            "missing_consumption"
        )
        .when(
            F.col("end_time") <= F.col("start_time"),
            "invalid_time_interval"
        )
        .when(
            F.col("dataset_id") != 193,
            "unexpected_dataset_id"
        )
        .when(
            F.col("consumption_mw") < 0,
            "negative_consumption"
        )
    )
    .withColumn(
        "quarantined_at",
        F.current_timestamp()
    )
)


In [0]:
display(validated_df)

Create separate dataframes for valid and invalid rows

In [0]:

silver_df = (
    validated_df
    .filter(F.col("validation_error").isNull())
    .drop("validation_error", "quarantined_at")
)

quarantine_df = (
    validated_df
    .filter(F.col("validation_error").isNotNull())
)

Check number of rows in both dataframes

In [0]:
silver_df.count()

In [0]:
quarantine_df.count()

Remove duplicates from valid dataframe

In [0]:
silver_df = silver_df.dropDuplicates(
    ["dataset_id", "start_time", "end_time"]
)

In [0]:
silver_df.count()

Save the data frames to tables in schema silver

In [0]:
silver_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        "energy.silver.consumption"
    )

In [0]:
quarantine_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        "energy.silver.consumption_quarantine"
    )

In [0]:
%sql

SELECT * FROM energy.silver.consumption LIMIT 20